# Import

In [170]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random

In [171]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Prep

## Loading variables

In [172]:
DISEASE = "BIPOLAR"
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")
    
    
sim_mat = load_npz(f"{DISEASE_FOLDER}/agg_sim_mat.npz")

In [173]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [174]:
# Loading result graph and communities
with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{RESULT_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{RESULT_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

## Helpful functions (big object, drop NAN)

In [175]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to HGNC

In [176]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi.get, c)) for c in comms]
    return comms_ncbi

In [177]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))

[['91966', '23522', '3964', '310', '54509', '8682', '55754', '3613', '10914', '8417', '23180', '23295', '9202', '51068', '23185', '25959', '51192', '7844', '81566', '10421', '23576', '85458', '4072', '26268', '29969', '1192', '9961', '10868', '9898', '258010', '55326', '55959', '162427', '10788', '259230', '79888', '9903', '11014', '6768', '64114', '5912', '55759', '81545', '55544', '55', '51136', '80237', '23673', '4430', '9221', '51762', '90355', '4212', '6675', '51199', '51322', '23160', '54542', '79699', '10950', '9847', '9590', '9522', '9891', '7289', '57187', '8612', '3159', '54464', '170622', '57222', '29775', '253782', '25921', '23111', '27042', '8624', '253943', '51768', '9482', '4281', '57403', '26353', '2029', '10664', '84301', '84525', '22868', '1656', '23307', '83786', '51747', '9980', '22848', '54832', '2107', '54465', '23051', '23197', '10920', '8444', '10572', '197131', '8706', '7072', '29946', '7205', '10960', '2971', '9559', '55323', '116068', '5867', '23473', '5874',

In [178]:
# NCBI to HGNC symbol
def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        mg = mygene.MyGeneInfo()
        entrez_ids = [str(e) for e in community]

        results = mg.querymany(
            entrez_ids,
            scopes="entrezgene",
            fields="symbol",
            species="human"
        )

        # Build a mapping: input ID -> symbol (or None)
        id_to_symbol = {}
        for r in results:
            q = str(r.get("query"))
            id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

        # Preserve original order
        symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
        comms_HGNC.append(symbols)
    return comms_HGNC


In [179]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
Input sequen

In [180]:
print(len(COMMUNITIES_HGNC))

14


In [181]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)

Community 0: dropped 0 NaN entries
Community 1: dropped 0 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 0 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries

Total dropped across all communities: 0


In [182]:
num_selected_comm = len(COMMUNITIES_HGNC)

In [183]:
print(num_selected_comm)

14


# Categoization Prep

### GO-slim

In [184]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [185]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [186]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [187]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [188]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [189]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [190]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=60)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

# Run Enrichment Analysis

In [191]:
TERM_SCORE_CAP = 0.001
PERCENTAGE = 0.1

### GO

In [192]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    category_counts_and_overlap_score_list = {}
    i = 0
    num_nonzero_communities = 0
    
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        

        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["GO_ID"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["GO_ID"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Compute overlap score for every category:
        filtered_exploded = filtered.explode('Category').reset_index(drop=True)
        category_counts_and_overlap_score = {}
        for val, group in filtered_exploded.groupby('Category'):
            overlap_list = group["Overlap"].tolist()
            numerators = [(lambda x: int(x.split("/")[0]))(e) for e in overlap_list]
            denominators = [(lambda x: int(x.split("/")[1]))(e) for e in overlap_list]
            overlap_score = sum(numerators)/sum(denominators)
            
            category_counts_and_overlap_score[val] = (len(group),overlap_score,)
        
        category_counts_and_overlap_score_list[i] = category_counts_and_overlap_score
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            print(category_counts_and_overlap_score)
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,category_counts_and_overlap_score_list

In [193]:
go_important_terms,go_category_counts_and_overlap_score = go_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE,slim_ids,depth = 1)

Size of community: 1135
Number of filtered terms: 51
Number of unmapped terms: 4
{'binding': (5, 0.12061902594446973), 'biological regulation': (4, 0.1880597014925373), 'catalytic activity': (6, 0.44715447154471544), 'cellular anatomical structure': (9, 0.11898496240601504), 'cellular process': (18, 0.1784251251706873), 'localization': (10, 0.15959952885747938), 'protein-containing complex': (2, 0.17318435754189945)}


C:\Users\celem\AppData\Local\Temp\ipykernel_22708\1851936443.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
3999,0,THO Complex Part Of Transcription Export Complex (GO:0000445),5/5,1.861280e-05,{GO:0032991},[protein-containing complex]
3354,0,Lysophospholipid Acyltransferase Activity (GO:0071617),6/8,8.592994e-05,{GO:0003824},[catalytic activity]
3359,0,2-Acylglycerol-3-Phosphate O-acyltransferase Activity (GO:0047144),5/7,6.496474e-04,{GO:0003824},[catalytic activity]
20,0,Positive Regulation Of rRNA Processing (GO:2000234),6/9,3.765766e-04,{GO:0065007},[biological regulation]
26,0,"Heparan Sulfate Proteoglycan Biosynthetic Process, Enzymatic Modification (GO:0015015)",6/10,6.977027e-04,{},[]
3351,0,Lysophosphatidic Acid Acyltransferase Activity (GO:0042171),11/20,4.194531e-07,{GO:0003824},[catalytic activity]
19,0,Heparan Sulfate Proteoglycan Metabolic Process (GO:0030201),7/13,3.765766e-04,{GO:0009987},[cellular process]
3352,0,1-Acylglycerol-3-Phosphate O-acyltransferase Activity (GO:0003841),10/19,3.086183e-06,{GO:0003824},[catalytic activity]
28,0,mRNA Methylation (GO:0080009),7/15,9.242862e-04,{},[]
17,0,Negative Regulation Of Macroautophagy (GO:0016242),9/22,2.793853e-04,{GO:0065007},[biological regulation]


Size of community: 828
Number of filtered terms: 5
Number of unmapped terms: 0
{'catalytic activity': (3, 0.1608832807570978), 'cellular process': (2, 0.1504424778761062)}


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
274,1,Cysteine-Type Deubiquitinase Activity (GO:0004843),17/98,0.000045,{GO:0003824},[catalytic activity]
275,1,Cysteine-Type Endopeptidase Activity (GO:0004197),17/106,0.000070,{GO:0003824},[catalytic activity]
0,1,Protein Deubiquitination (GO:0016579),17/112,0.000615,{GO:0009987},[cellular process]
276,1,Deubiquitinase Activity (GO:0101005),17/113,0.000115,{GO:0003824},[catalytic activity]
1,1,Protein Modification By Small Protein Removal (GO:0070646),17/114,0.000615,{GO:0009987},[cellular process]


...

,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
1,9,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),70/141,2.375157e-101,{GO:0050896},[response to stimulus]
2,9,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),69/139,5.322928e-100,{GO:0050896},[response to stimulus]
0,9,Sensory Perception Of Smell (GO:0007608),112/230,1.570768e-166,{GO:0032501},[multicellular organismal process]
3,9,Sensory Perception Of Chemical Stimulus (GO:0007606),53/110,3.298646e-75,{GO:0032501},[multicellular organismal process]
17,9,Olfactory Receptor Activity (GO:0004984),174/362,1.243766e-278,{GO:0060089},[molecular transducer activity]


Size of community: 89
Number of filtered terms: 4
Number of unmapped terms: 0
{'transporter activity': (4, 0.21052631578947367)}


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
118,12,Potassium:Proton Antiporter Activity (GO:0015386),3/12,0.000618,{GO:0005215},[transporter activity]
119,12,Sodium:Proton Antiporter Activity (GO:0015385),3/14,0.000618,{GO:0005215},[transporter activity]
120,12,Solute:Inorganic Anion Antiporter Activity (GO:0005452),3/14,0.000618,{GO:0005215},[transporter activity]
121,12,Solute:Potassium Antiporter Activity (GO:0022821),3/17,0.000858,{GO:0005215},[transporter activity]


8 out of 14 communities had significant GO terms.


c:\Users\celem\AppData\Local\Programs\Python\Python310\lib\site-packages\gseapy\enrichr.py:689: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.results = pd.concat(self.results, ignore_index=True)


In [194]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value)
0,0,1135,THO Complex Part Of Transcription Export Compl...,5/5,1.861280e-05,[protein-containing complex],GO_Cellular_Component_2023,5.836767e-07,0.0,0.0,94325.000000,1.353933e+06,THOC3;THOC2;THOC5;THOC7;THOC6,GO:0000445,{GO:0032991},1.000000
1,0,1135,Lysophospholipid Acyltransferase Activity (GO:...,6/8,8.592994e-05,[catalytic activity],GO_Molecular_Function_2023,8.364117e-07,0.0,0.0,50.123118,7.014302e+02,LPCAT3;MBOAT7;LPCAT1;PNPLA3;MBOAT2;ABHD5,GO:0071617,{GO:0003824},0.750000
2,0,1135,2-Acylglycerol-3-Phosphate O-acyltransferase A...,5/7,6.496474e-04,[catalytic activity],GO_Molecular_Function_2023,1.113103e-05,0.0,0.0,41.732301,4.759892e+02,LPCAT3;MBOAT7;LPCAT1;MBOAT1;MBOAT2,GO:0047144,{GO:0003824},0.714286
3,0,1135,Positive Regulation Of rRNA Processing (GO:200...,6/9,3.765766e-04,[biological regulation],GO_Biological_Process_2023,2.388067e-06,0.0,0.0,33.413640,4.325405e+02,DIMT1;HEATR1;WDR75;RIOK2;RIOK1;WDR43,GO:2000234,{GO:0065007},0.666667
4,0,1135,Heparan Sulfate Proteoglycan Biosynthetic Proc...,6/10,6.977027e-04,[],GO_Biological_Process_2023,5.682157e-06,0.0,0.0,25.058902,3.026659e+02,HS3ST3B1;NDST2;NDST1;HS3ST3A1;HS6ST1;HS6ST2,GO:0015015,{},0.600000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,9,219,Olfactory Receptor Activity (GO:0004984),174/362,1.243766e-278,[molecular transducer activity],GO_Molecular_Function_2023,1.243766e-278,0.0,0.0,402.976596,2.578649e+05,OR1C1;OR2M5;OR52N1;OR2M4;OR2T12;OR2T10;OR10AC1...,GO:0004984,{GO:0060089},0.480663
896,12,89,Potassium:Proton Antiporter Activity (GO:0015386),3/12,6.184377e-04,[transporter activity],GO_Molecular_Function_2023,1.820313e-05,0.0,0.0,77.139535,8.418945e+02,SLC9A4;SLC9A5;SLC9A8,GO:0015386,{GO:0005215},0.250000
897,12,89,Sodium:Proton Antiporter Activity (GO:0015385),3/14,6.184377e-04,[transporter activity],GO_Molecular_Function_2023,2.992440e-05,0.0,0.0,63.107822,6.573839e+02,SLC9A4;SLC9A5;SLC9A8,GO:0015385,{GO:0005215},0.214286
898,12,89,Solute:Inorganic Anion Antiporter Activity (GO...,3/14,6.184377e-04,[transporter activity],GO_Molecular_Function_2023,2.992440e-05,0.0,0.0,63.107822,6.573839e+02,SLC4A10;SLC26A4;SLC26A3,GO:0005452,{GO:0005215},0.214286


### KEGG

In [195]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    category_counts_and_overlap_score_list = {}
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Compute overlap score for every category:
        filtered_exploded = filtered.explode('Category').reset_index(drop=True)
        category_counts_and_overlap_score = {}
        for val, group in filtered_exploded.groupby('Category'):
            overlap_list = group["Overlap"].tolist()
            numerators = [(lambda x: int(x.split("/")[0]))(e) for e in overlap_list]
            denominators = [(lambda x: int(x.split("/")[1]))(e) for e in overlap_list]
            overlap_score = sum(numerators)/sum(denominators)
            
            category_counts_and_overlap_score[val] = (len(group),overlap_score,)
        
        category_counts_and_overlap_score_list[i] = category_counts_and_overlap_score
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            print(category_counts_and_overlap_score)
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,category_counts_and_overlap_score_list

In [196]:
kegg_important_terms,kegg_category_counts_and_overlap_score = kegg_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

Size of community: 1135
Number of filtered terms: 7


C:\Users\celem\AppData\Local\Temp\ipykernel_22708\3460503351.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,0,Glycosaminoglycan biosynthesis,21/53,6.831257e-11,NaN,[]
3,0,Mucin type O-glycan biosynthesis,12/36,1.997033e-05,hsa00512,[Glycan biosynthesis and metabolism]
2,0,Sphingolipid metabolism,14/49,1.997033e-05,hsa00600,[Lipid metabolism]
6,0,Glycosphingolipid biosynthesis,11/45,9.838766e-04,NaN,[]
5,0,N-Glycan biosynthesis,12/50,6.172302e-04,hsa00510,[Glycan biosynthesis and metabolism]
4,0,Glycerolipid metabolism,14/61,2.622493e-04,hsa00561,[Lipid metabolism]
1,0,RNA transport,30/186,1.997033e-05,NaN,[]


{'Glycan biosynthesis and metabolism': (2, 0.27906976744186046), 'Lipid metabolism': (2, 0.2545454545454545)}
Size of community: 1080
Number of filtered terms: 31


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,2,Spliceosome,66/150,2.969117e-41,hsa03040,[Transcription]
11,2,RNA polymerase,11/31,6.265556e-06,hsa03020,[Transcription]
1,2,Ubiquitin mediated proteolysis,39/140,7.577364e-16,hsa04120,"[Folding, sorting and degradation]"
3,2,RNA degradation,22/79,6.009886e-09,hsa03018,"[Folding, sorting and degradation]"
14,2,Nucleotide excision repair,13/47,1.152983e-05,hsa03420,[Replication and repair]
27,2,Basal transcription factors,11/45,1.603386e-04,hsa03022,[Transcription]
17,2,Basal cell carcinoma,15/63,1.152983e-05,hsa05217,[Cancer: specific types]
16,2,Adherens junction,16/71,1.152983e-05,hsa04520,[Cellular community - eukaryotes]
24,2,Arrhythmogenic right ventricular cardiomyopathy,15/77,1.179382e-04,hsa05412,[Cardiovascular disease]
20,2,ECM-receptor interaction,17/88,4.108985e-05,hsa04512,[Signaling molecules and interaction]


...

{'Translation': (1, 0.23417721518987342)}
Size of community: 515
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,7,Neuroactive ligand-receptor interaction,73/341,4.500366e-44,hsa04080,[Signaling molecules and interaction]


{'Signaling molecules and interaction': (1, 0.21407624633431085)}
Size of community: 219
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,9,Olfactory transduction,214/440,0.0,hsa04740,[Sensory system]


{'Sensory system': (1, 0.4863636363636364)}
5 out of 14 communities had significant GO terms.


In [197]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,KEGG_ID,Overlap (value)
0,0,1135,Glycosaminoglycan biosynthesis,21/53,6.831257e-11,[],KEGG_2021_Human,3.162619e-13,0.0,0.0,11.094395,319.321165,HS3ST3B1;CHST7;HS3ST3A1;GLCE;CSGALNACT2;HS6ST1...,NaN,0.396226
1,0,1135,Mucin type O-glycan biosynthesis,12/36,1.997033e-05,[Glycan biosynthesis and metabolism],KEGG_2021_Human,3.698209e-07,0.0,0.0,8.388691,124.238587,GALNT12;GALNT7;GALNT11;GALNT14;GALNT5;ST6GALNA...,hsa00512,0.333333
2,0,1135,Sphingolipid metabolism,14/49,1.997033e-05,[Lipid metabolism],KEGG_2021_Human,3.449416e-07,0.0,0.0,6.719001,99.977999,CERS4;CERS6;CERK;SGMS1;SPHK2;SGPP2;NEU3;SPTLC1...,hsa00600,0.285714
3,0,1135,Glycosphingolipid biosynthesis,11/45,9.838766e-04,[],KEGG_2021_Human,3.188489e-05,0.0,0.0,5.420269,56.118095,B3GALNT1;ST8SIA1;B3GALT4;B3GNT3;B3GNT2;B4GALNT...,NaN,0.244444
4,0,1135,N-Glycan biosynthesis,12/50,6.172302e-04,[Glycan biosynthesis and metabolism],KEGG_2021_Human,1.714528e-05,0.0,0.0,5.294184,58.097248,FUT8;GANAB;ST6GAL2;MAN2A2;MAN2A1;MGAT5;MGAT5B;...,hsa00510,0.240000
5,0,1135,Glycerolipid metabolism,14/61,2.622493e-04,[Lipid metabolism],KEGG_2021_Human,6.070585e-06,0.0,0.0,5.000323,60.064154,AGPAT5;DGKD;GK;DGKA;MBOAT1;MBOAT2;AGPAT3;AGPAT...,hsa00561,0.229508
6,0,1135,RNA transport,30/186,1.997033e-05,[],KEGG_2021_Human,2.120710e-07,0.0,0.0,3.256004,50.032883,NUP205;DDX20;NMD3;NXF1;EIF2B1;RAE1;NDC1;EIF5B;...,NaN,0.161290
7,2,1080,Spliceosome,66/150,2.969117e-41,[Transcription],KEGG_2021_Human,1.331443e-43,0.0,0.0,14.595379,1440.927271,RBM25;EIF4A3;HNRNPU;PRPF19;PQBP1;EFTUD2;SNRPD2...,hsa03040,0.440000
8,2,1080,RNA polymerase,11/31,6.265556e-06,[Transcription],KEGG_2021_Human,3.371599e-07,0.0,0.0,9.724041,144.914550,POLR2B;POLR2C;POLR2D;POLR2E;POLR2F;POLR2G;POLR...,hsa03020,0.354839
9,2,1080,Ubiquitin mediated proteolysis,39/140,7.577364e-16,"[Folding, sorting and degradation]",KEGG_2021_Human,6.795842e-18,0.0,0.0,6.980540,275.942303,UBA6;UBE2D2;UBE2D3;NEDD4L;UBE2L6;PRPF19;RCHY1;...,hsa04120,0.278571


### Reactome

In [198]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    category_counts_and_overlap_score_list= {}
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        reactome_level1 = build_reactome_level_map(level = 1)
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Compute overlap score for every category:
        filtered_exploded = filtered.explode('Category').reset_index(drop=True)
        category_counts_and_overlap_score = {}
        for val, group in filtered_exploded.groupby('Category'):
            overlap_list = group["Overlap"].tolist()
            numerators = [(lambda x: int(x.split("/")[0]))(e) for e in overlap_list]
            denominators = [(lambda x: int(x.split("/")[1]))(e) for e in overlap_list]
            overlap_score = sum(numerators)/sum(denominators)
            
            category_counts_and_overlap_score[val] = (len(group),overlap_score,)
        
        category_counts_and_overlap_score_list[i] = category_counts_and_overlap_score
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            print(category_counts_and_overlap_score)
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms,category_counts_and_overlap_score_list

In [199]:
reactome_important_terms,reactome_category_counts_and_overlap_score = reactome_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

https://reactome.org/ContentService/data/eventsHierarchy/9606
Size of community: 1135
Number of filtered terms: 16


C:\Users\celem\AppData\Local\Temp\ipykernel_22708\2239647310.py:49: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
8,0,HS-GAG Biosynthesis R-HSA-2022928,11/30,3.594505e-05,[Metabolism]
4,0,mRNA 3-End Processing R-HSA-72187,16/58,1.486605e-05,[Metabolism of RNA]
7,0,rRNA Modification In Nucleus And Cytosol R-HSA-6790901,16/60,1.686702e-05,[Metabolism of RNA]
12,0,RNA Polymerase II Transcription Termination R-HSA-73856,16/67,5.354529e-05,[Gene expression (Transcription)]
10,0,Transport Of Mature mRNA Derived From An Intron-Containing Transcript R-HSA-159236,17/74,5.191137e-05,[Metabolism of RNA]
11,0,Transport Of Mature Transcript To Cytoplasm R-HSA-72202,18/83,5.354529e-05,[Metabolism of RNA]
3,0,Glycosaminoglycan Metabolism R-HSA-1630316,25/120,2.912919e-06,[Metabolism]
14,0,Sphingolipid Metabolism R-HSA-428157,17/89,5.307554e-04,[Metabolism]
15,0,Glycerophospholipid Biosynthesis R-HSA-1483206,21/127,5.307554e-04,[Metabolism]
9,0,Intra-Golgi And Retrograde Golgi-to-ER Traffic R-HSA-6811442,29/181,3.594505e-05,[Vesicle-mediated transport]


{'Gene expression (Transcription)': (1, 0.23880597014925373), 'Immune System': (1, 0.11375661375661375), 'Metabolism': (5, 0.17511520737327188), 'Metabolism of RNA': (6, 0.15638207945900254), 'Metabolism of proteins': (1, 0.12411347517730496), 'Vesicle-mediated transport': (2, 0.11794871794871795)}
https://reactome.org/ContentService/data/eventsHierarchy/9606
https://reactome.org/ContentService/data/eventsHierarchy/9606
Size of community: 1080
Number of filtered terms: 161


,Community Index,Term,Overlap,Adjusted P-value,Category
13,2,Signaling By FGFR2 IIIa TM R-HSA-8851708,16/19,3.119363e-16,[Disease]
71,2,Folding Of Actin By CCT/TriC R-HSA-390450,8/10,4.693896e-08,[Metabolism of proteins]
94,2,SLBP Independent Processing Of Histone Pre-mRNAs R-HSA-111367,7/10,1.687348e-06,[Metabolism of RNA]
10,2,mRNA Capping R-HSA-72086,20/29,2.388285e-17,[Metabolism of RNA]
162,2,WNT Mediated Activation Of DVL R-HSA-201688,4/6,8.368735e-04,[Signal Transduction]
161,2,Fibronectin Matrix Formation R-HSA-1566977,4/6,8.368735e-04,[Extracellular matrix organization]
21,2,RNA Pol II CTD Phosphorylation And Interaction With CE R-HSA-77075,18/27,2.091855e-15,[Gene expression (Transcription)]
28,2,FGFR2 Alternative Splicing R-HSA-6803529,17/26,1.991131e-14,[Signal Transduction]
6,2,mRNA Splicing - Minor Pathway R-HSA-72165,32/49,7.886120e-27,[Metabolism of RNA]
38,2,Abortive Elongation Of HIV-1 Transcript In Absence Of Tat R-HSA-167242,15/23,8.643492e-13,[Disease]


...

{'Disease': (2, 0.14864864864864866), 'Signal Transduction': (17, 0.20815933183424348)}
https://reactome.org/ContentService/data/eventsHierarchy/9606
https://reactome.org/ContentService/data/eventsHierarchy/9606
Size of community: 219
Number of filtered terms: 3


,Community Index,Term,Overlap,Adjusted P-value,Category
1,9,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,206/393,0.000000e+00,[Sensory Perception]
0,9,Olfactory Signaling Pathway R-HSA-381753,206/401,0.000000e+00,[Sensory Perception]
2,9,Sensory Perception R-HSA-9709957,206/616,2.912586e-308,[Sensory Perception]


{'Sensory Perception': (3, 0.43829787234042555)}
https://reactome.org/ContentService/data/eventsHierarchy/9606
https://reactome.org/ContentService/data/eventsHierarchy/9606
https://reactome.org/ContentService/data/eventsHierarchy/9606
Size of community: 89
Number of filtered terms: 5


,Community Index,Term,Overlap,Adjusted P-value,Category
5,12,Sodium/Proton Exchangers R-HSA-425986,3/8,3.989604e-05,[Transport of small molecules]
6,12,Bicarbonate Transporters R-HSA-425381,3/10,7.280756e-05,[Transport of small molecules]
0,12,Beta Defensins R-HSA-1461957,9/35,1.478938e-12,[Immune System]
1,12,Defensins R-HSA-1461973,9/43,5.738189e-12,[Immune System]
2,12,Antimicrobial Peptides R-HSA-6803157,9/89,3.652790e-09,[Immune System]


{'Immune System': (3, 0.16167664670658682), 'Transport of small molecules': (2, 0.3333333333333333)}
https://reactome.org/ContentService/data/eventsHierarchy/9606
8 out of 14 communities had significant GO terms.


In [200]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1135,HS-GAG Biosynthesis R-HSA-2022928,11/30,3.594505e-05,[Metabolism],Reactome_2022,3.765324e-07,0.0,0.0,9.707155,143.590776,NDST2;HS3ST3B1;NDST1;HS3ST3A1;GLCE;SDC1;GPC4;A...,0.366667
1,0,1135,mRNA 3-End Processing R-HSA-72187,16/58,1.486605e-05,[Metabolism of RNA],Reactome_2022,8.681102e-08,0.0,0.0,6.408102,104.192749,FYTTD1;CHTOP;CPSF7;CPSF6;CPSF1;POLDIP3;SRSF1;U...,0.275862
2,0,1135,rRNA Modification In Nucleus And Cytosol R-HSA...,16/60,1.686702e-05,[Metabolism of RNA],Reactome_2022,1.458770e-07,0.0,0.0,6.116175,96.271669,UTP25;DDX47;DIMT1;SPPL2A;HEATR1;WDR75;WDR43;RR...,0.266667
3,0,1135,RNA Polymerase II Transcription Termination R-...,16/67,5.354529e-05,[Gene expression (Transcription)],Reactome_2022,7.525284e-07,0.0,0.0,5.274738,74.372888,FYTTD1;CHTOP;CPSF7;CPSF6;CPSF1;POLDIP3;SRSF1;U...,0.238806
4,0,1135,Transport Of Mature mRNA Derived From An Intro...,17/74,5.191137e-05,[Metabolism of RNA],Reactome_2022,6.173244e-07,0.0,0.0,5.017356,71.737504,NDC1;FYTTD1;NUP205;CHTOP;POLDIP3;SRSF1;UPF3B;T...,0.229730
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278,12,89,Sodium/Proton Exchangers R-HSA-425986,3/8,3.989604e-05,[Transport of small molecules],Reactome_2022,4.693652e-06,0.0,0.0,138.879070,1703.948928,SLC9A4;SLC9A5;SLC9A8,0.375000
279,12,89,Bicarbonate Transporters R-HSA-425381,3/10,7.280756e-05,[Transport of small molecules],Reactome_2022,9.993194e-06,0.0,0.0,99.189369,1142.027342,SLC4A9;SLC4A10;SLC4A5,0.300000
280,12,89,Beta Defensins R-HSA-1461957,9/35,1.478938e-12,[Immune System],Reactome_2022,2.899879e-14,0.0,0.0,86.040865,2682.024755,DEFB105A;DEFB119;DEFB129;DEFB125;DEFB136;DEFB1...,0.257143
281,12,89,Defensins R-HSA-1461973,9/43,5.738189e-12,[Immune System],Reactome_2022,2.250270e-13,0.0,0.0,65.769485,1915.375512,DEFB105A;DEFB119;DEFB129;DEFB125;DEFB136;DEFB1...,0.209302


### Disease Data Sets

In [201]:
# disease_term_score_cap = 0.001
# disease_percentage = 0.1
# important_diseases = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value"])

In [202]:
# # Disease-gene enrichment libraries
# disease_sets = [
#     'DisGeNET_2020', # curated gene–disease associations
#     'GWAS_Catalog_2023', # genome-wide association hits
#     'OMIM_Disease', # Mendelian disorders
#     'Jensen_DISEASES' # text-mined associations
# ]

# # # Disease-gene enrichment Analysis; save terms with small size and high p-value
# i = 0
# for community in communities_HGNC:
#     # Gene Ontology enrichment
#     enr_disease = gp.enrichr(
#         gene_list=community,
#         gene_sets=disease_sets,
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     enr_disease_df = enr_disease.results.sort_values('Adjusted P-value')
#     print(f"Size of community: {len(community)}")

#     mask =  (enr_disease_df["Adjusted P-value"] < disease_term_score_cap) & (enr_disease_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > disease_percentage))
        
#     filtered = enr_disease_df[mask].copy()
#     if not filtered.empty:
#         filtered.loc[:, "Community Index"] = i
#         filtered.loc[:, "Community Size"] = len(community)
#         important_diseases = pd.concat([important_diseases, filtered], ignore_index=True)

#     display(HTML(filtered[['Term','Overlap','Adjusted P-value']].head(10).to_html(max_cols=None)))
#     i += 1

# Important Terms Analysis

### Constructing Important Terms df

In [157]:
def comm_similarity_with_term(x,y):
    return 1-(abs(x-y)/max(x,y))

In [158]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])

In [159]:
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)

In [160]:
# important_terms = important_terms.sort_values(by="Overlap (value)",ascending=False)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
0,0,1133,THO Complex Part Of Transcription Export Compl...,5/5,1.844910e-05,[protein-containing complex],GO_Cellular_Component_2023,5.785432e-07,0.0,0.0,94335.000000,1.354910e+06,THOC3;THOC2;THOC5;THOC7;THOC6,GO:0000445,{GO:0032991},1.000000,NaN
935,0,1133,HS-GAG Biosynthesis R-HSA-2022928,11/30,3.451050e-05,[Metabolism],Reactome_2022,3.699402e-07,0.0,0.0,9.725490,1.440338e+02,NDST2;HS3ST3B1;NDST1;HS3ST3A1;GLCE;SDC1;GPC4;A...,NaN,NaN,0.366667,NaN
53,0,1133,RNA Binding (GO:0003723),143/1411,2.487569e-09,[binding],GO_Molecular_Function_2023,3.880763e-12,0.0,0.0,2.004793,5.267592e+01,OTUD4;TCERG1;RPL31;CISD2;NOC2L;MKI67;RRP8;ALKB...,GO:0003723,{GO:0005488},0.101347,NaN
52,0,1133,Lysosome (GO:0005764),51/503,8.539676e-04,[cellular anatomical structure],GO_Cellular_Component_2023,4.165696e-05,0.0,0.0,1.920331,1.936854e+01,SPPL2A;VLDLR;ATRAID;CHMP1B;CTSK;ANKFY1;ACP3;AP...,GO:0005764,{GO:0110165},0.101392,NaN
51,0,1133,Intracellular Non-Membrane-Bounded Organelle (...,122/1195,6.440591e-09,[cellular anatomical structure],GO_Cellular_Component_2023,1.122054e-10,0.0,0.0,2.001163,4.584803e+01,MAPKBP1;FHOD1;NOC2L;MKI67;RRP8;RRP9;SMC2;CDC20...,GO:0043232,{GO:0110165},0.102092,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1218,12,89,Defensins R-HSA-1461973,10/43,7.836887e-14,[Immune System],Reactome_2022,3.073289e-15,0.0,0.0,76.248562,2.547924e+03,DEFB105A;DEFB119;DEFB129;DEFB125;DEFB136;DEFB1...,NaN,NaN,0.232558,NaN
1215,12,89,Sodium/Proton Exchangers R-HSA-425986,3/8,3.989604e-05,[Transport of small molecules],Reactome_2022,4.693652e-06,0.0,0.0,138.879070,1.703949e+03,SLC9A4;SLC9A5;SLC9A8,NaN,NaN,0.375000,NaN
1216,12,89,Bicarbonate Transporters R-HSA-425381,3/10,7.280756e-05,[Transport of small molecules],Reactome_2022,9.993194e-06,0.0,0.0,99.189369,1.142027e+03,SLC4A9;SLC4A10;SLC4A5,NaN,NaN,0.300000,NaN
1217,12,89,Beta Defensins R-HSA-1461957,10/35,1.544461e-14,[Immune System],Reactome_2022,3.028355e-16,0.0,0.0,100.688608,3.597940e+03,DEFB105A;DEFB119;DEFB129;DEFB125;DEFB136;DEFB1...,NaN,NaN,0.285714,NaN


In [161]:
# Community id to size dict
com_id_to_size = {i : len(COMMUNITIES_HGNC[i]) for i in range(len(COMMUNITIES_HGNC))}

In [162]:
unique_com_id_to_size = important_terms.drop_duplicates(subset="Community Index", keep="first")

In [163]:
comm_size_dict = dict(zip(unique_com_id_to_size["Community Index"], unique_com_id_to_size["Community Size"]))

In [164]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms_{DISEASE}.csv", index=False)

### Graph Building

In [ ]:
# # df must have: "Community Index", "Term", "Overlap (value)"
# work = important_terms.loc[:, ["Community Index", "Term", "Overlap (value)","Category"]].copy()
# work["Overlap (value)"] = work["Overlap (value)"].astype(float)

# # Ensure (community, term) uniqueness
# dupes = work.duplicated(subset=["Community Index", "Term"], keep=False)
# if dupes.any():
#     raise ValueError("Duplicated (Community Index, Term) rows found; ensure uniqueness first.")

# # --- Build edge weights AND collect contributing terms per pair ---
# edge_weights = {}              # (u, v) -> float
# edge_counts  = {}              # (u, v) -> int
# edge_terms   = {}              # (u, v) -> list[(term, contrib)]

# for term, sub in work.groupby("Term", sort=False):
#     comms  = sub["Community Index"].to_numpy()
#     scores = sub["Overlap (value)"].to_numpy()
#     if len(comms) < 2:
#         continue
#     for i, j in combinations(range(len(comms)), 2):
#         u, v = comms[i], comms[j]
#         if u > v: u, v = v, u  # canonical ordering
#         contrib = comm_similarity_with_term(scores[i], scores[j])

#         edge_weights[(u, v)] = edge_weights.get((u, v), 0.0) + contrib
#         edge_counts[(u, v)]  = edge_counts.get((u, v), 0)    + 1
#         edge_terms.setdefault((u, v), []).append((term, contrib))

# # Sort contributing terms by contribution desc for each edge
# for key in edge_terms:
#     edge_terms[key].sort(key=lambda t: t[1], reverse=True)

# # --- Build edge list DataFrame (optional, useful to inspect) ---
# edge_df = pd.DataFrame(
#     [(u, v, edge_weights[(u, v)], edge_counts[(u, v)], edge_terms.get((u, v), []))
#      for (u, v) in edge_weights.keys()],
#     columns=["u", "v", "weight", "shared_terms", "terms_contrib"]
# ).sort_values(["weight", "shared_terms"], ascending=[False, False]).reset_index(drop=True)

# # --- Build NetworkX graph with attributes ---
# G = nx.Graph()
# G.add_nodes_from(pd.unique(work["Community Index"]))
# for _, r in edge_df.iterrows():
#     G.add_edge(
#         int(r.u), int(r.v),
#         weight=float(r.weight),
#         shared_terms=int(r.shared_terms),
#         terms_contrib=r.terms_contrib  # list of (term, contrib) sorted desc
#     )

In [ ]:
work

### Table

In [ ]:
# #--------------Table------------------
# term_contribs = []

# for term, sub in work.groupby("Term", sort=False):
#     comms  = sub["Community Index"].to_numpy()
#     scores = sub["Overlap (value)"].to_numpy()
#     if len(comms) < 2:
#         continue
#     for i, j in combinations(range(len(comms)), 2):
#         u, v = comms[i], comms[j]
#         if u > v:
#             u, v = v, u
#         contrib = comm_similarity_with_term(scores[i], scores[j])
#         cat = sub["Category"].iloc[0] if "Category" in sub.columns else None
#         term_contribs.append((u, v, term, contrib, cat))

# # 2) Build DataFrame
# term_df = pd.DataFrame(term_contribs, columns=["u", "v", "Term", "Contribution","Category"])
# # 3) Sort and aggregate terms per edge (keep per-term order)
# agg_blocks = []
# for (u, v), sub in term_df.groupby(["u", "v"]):
#     sub_sorted = sub.sort_values("Contribution", ascending=False)
    
#     # Create category count dictionary
#     category_counts = Counter(
#         c
#         for cats in sub_sorted["Category"].dropna()
#         for c in cats
#     )
#     category_counts_dict = dict(category_counts)

#     # sub_sorted = sub.sort_values(sub_sorted["Category"].apply(tuple), ascending=False)
#     block = "\n".join(
#         [f"  - {t} {cat} ({c:.3f})"
#         for t, c, cat in zip(sub_sorted["Term"], sub_sorted["Contribution"], sub_sorted["Category"])]
#     )
#     total = sub_sorted["Contribution"].sum()
#     agg_blocks.append({
#         "u": u,
#         "v": v,
#         "Total Weight": total,
#         "Terms (by contribution)": block,
#         "Category Count": category_counts_dict
#     })

# # 4) Create final block table
# block_df = pd.DataFrame(agg_blocks).sort_values("Total Weight", ascending=False).reset_index(drop=True)
# # 5) Display nicely
# for _, row in block_df.iterrows():
#     print(f"Community pair ({row.u}, {row.v}) — Total Weight = {row['Total Weight']:.3f}")
#     print(row["Terms (by contribution)"])
    
#     print()
#     print("Category Count:")
#     for key, value in sorted(row["Category Count"].items(), key=lambda x: x[1], reverse=True):
#         print(f"{key}: {value}")

#     print("-" * 60)

# Category Counts

In [203]:
def print_category_count_by_comm(category_count_by_comm):
    for comm_id, cat_dict in category_count_by_comm.items():
        print(f"\n🧩 Community {comm_id}")
        print("-" * (14 + len(str(comm_id))))

        if not cat_dict:
            print("  (no categories)")
            continue

        # Sort categories by descending count
        for cat, count in sorted(cat_dict.items(), key=lambda x: x[1], reverse=True):
            print(f"  • {cat:<50} {count}")

In [204]:
category_count_by_comm = {}
for i in range(num_selected_comm):
    comm_cates = go_category_counts_and_overlap_score[i] | kegg_category_counts_and_overlap_score[i] | reactome_category_counts_and_overlap_score[i]
    category_count_by_comm[i] = dict(sorted(comm_cates.items(), key=lambda x: x[1],reverse=True))

In [205]:
category_count_by_comm

{0: {'cellular process': (18, 0.1784251251706873),
  'localization': (10, 0.15959952885747938),
  'cellular anatomical structure': (9, 0.11898496240601504),
  'catalytic activity': (6, 0.44715447154471544),
  'Metabolism of RNA': (6, 0.15638207945900254),
  'Metabolism': (5, 0.17511520737327188),
  'binding': (5, 0.12061902594446973),
  'biological regulation': (4, 0.1880597014925373),
  'Glycan biosynthesis and metabolism': (2, 0.27906976744186046),
  'Lipid metabolism': (2, 0.2545454545454545),
  'protein-containing complex': (2, 0.17318435754189945),
  'Vesicle-mediated transport': (2, 0.11794871794871795),
  'Gene expression (Transcription)': (1, 0.23880597014925373),
  'Metabolism of proteins': (1, 0.12411347517730496),
  'Immune System': (1, 0.11375661375661375)},
 1: {'catalytic activity': (3, 0.1608832807570978),
  'cellular process': (2, 0.1504424778761062)},
 2: {'biological regulation': (378, 0.2068401592718999),
  'cellular process': (192, 0.2062780269058296),
  'binding': 

In [206]:
with open(f"{DISEASE_FOLDER}/category_count_by_comm.pkl", "wb") as f:
    pickle.dump(category_count_by_comm, f)

In [207]:
print_category_count_by_comm(category_count_by_comm)


🧩 Community 0
---------------
  • cellular process                                   (18, 0.1784251251706873)
  • localization                                       (10, 0.15959952885747938)
  • cellular anatomical structure                      (9, 0.11898496240601504)
  • catalytic activity                                 (6, 0.44715447154471544)
  • Metabolism of RNA                                  (6, 0.15638207945900254)
  • Metabolism                                         (5, 0.17511520737327188)
  • binding                                            (5, 0.12061902594446973)
  • biological regulation                              (4, 0.1880597014925373)
  • Glycan biosynthesis and metabolism                 (2, 0.27906976744186046)
  • Lipid metabolism                                   (2, 0.2545454545454545)
  • protein-containing complex                         (2, 0.17318435754189945)
  • Vesicle-mediated transport                         (2, 0.11794871794871795)
  • Gene e

# Robustness Analysis

In [ ]:
def run_enrichment_func(community,term_score_cap,percentage):
    # GO df
    enr_go = gp.enrichr(
        gene_list=community,
        gene_sets=['GO_Biological_Process_2023',
                'GO_Molecular_Function_2023',
                'GO_Cellular_Component_2023'],
        organism='Human',
        outdir=None # don't write to disk
    )
    GO_df = enr_go.results
    mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
    GO_df = GO_df[mask].copy()   
    
    # KEGG df
    enr_kegg = gp.enrichr(
        gene_list=community,
        gene_sets=['KEGG_2021_Human'],
        organism='Human',
        outdir=None
    )
    KEGG_df = enr_kegg.results
    mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
    KEGG_df = KEGG_df[mask].copy() 
       
    # Reactome df
    enr_reactome = gp.enrichr(
        gene_list=community,
        gene_sets=['Reactome_2022'],
        organism='Human',
        outdir=None
    )
    Reactome_df = enr_reactome.results  
    mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
    Reactome_df = Reactome_df[mask].copy()
    
    
    all_df = [GO_df,KEGG_df,Reactome_df]
    # build result df by concatenating
    result = pd.concat(all_df, ignore_index=True)
    return result

In [ ]:
from json import JSONDecodeError

# ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
_ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
    """
    Calls user's run_enrichment_func(community) with retries + memoization.
    Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
    """
    # Ensure we always pass a list of gene symbols (never a bare string)
    genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
    if len(genes) == 0:
        return pd.DataFrame()

    key = tuple(sorted(genes))
    if key in _ENR_CACHE:
        return _ENR_CACHE[key].copy()

    for a in range(retries):
        try:
            df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
            if df is None:
                # treat as transient failure to trigger retry
                raise RuntimeError("run_enrichment_func returned None")
            _ENR_CACHE[key] = df.copy()
            return df
        except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
            # Transient errors from HTTP/JSON/file handling inside gseapy
            if a == retries - 1:
                # Give up: return empty so pipeline continues
                return pd.DataFrame()
            time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

    return pd.DataFrame()

# ---------------- 2) Minimal bootstrap to record robust terms ----------------
def get_robust_terms(communities_HGNC, run_enrichment_func,
                     R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
    """
    Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
    Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
    """
    rng = np.random.default_rng(seed)
    rows = []

    for cid, community in enumerate(communities_HGNC):
        n = len(community)
        if n == 0:
            continue
        drop_k = max(1, int(np.floor(leaveout * n)))
        counts = Counter()

        for _ in range(R):
            # Jackknife subset (ensure not empty)
            keep = np.ones(n, dtype=bool)
            keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
            sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
            if len(sub) == 0:
                continue

            df = run_enrichment_safe(run_enrichment_func, sub)
            if df is None or df.empty:
                continue

            # Your function already returns significant terms; just count them.
            # If it includes multiple libraries, preserve Gene_set to disambiguate names.
            if 'Term' not in df.columns:
                continue  # be defensive

            if 'Gene_set' in df.columns:
                terms = (df[['Term', 'Gene_set']]
                         .dropna()
                         .drop_duplicates()
                         .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
                         .tolist())
            else:
                terms = df['Term'].dropna().drop_duplicates().tolist()

            counts.update(terms)

            # tiny pause helps with API rate limits if your func calls Enrichr internally
            time.sleep(0.03)

        # Keep only robust terms
        for t, c in counts.items():
            freq = c / max(R, 1)
            if freq >= recurrence_cutoff:
                if '|' in t:
                    term, gene_set = t.split('|', 1)
                    rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
                else:
                    rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

    return (pd.DataFrame(rows)
              .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
              .reset_index(drop=True))

In [ ]:
twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
                                R=25, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
twr3

In [ ]:
terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
                                R=10, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
terms_with_recurrence

In [ ]:
# rename important terms to match terms_with_recurrence
important_terms = important_terms.rename(columns={'index': 'community_id'})
important_terms = important_terms.rename(columns={'Term': 'term'})

In [ ]:
terms_with_rec_merged = important_terms.merge(
    terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
    on=['community_id', 'term', 'Gene_set'],
    how='left'
)

terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

terms_with_rec_merged = terms_with_rec_merged.sort_values(
    ['community_id', 'recurrence'],
    ascending=[True, False]
).reset_index(drop=True)

In [ ]:
terms_with_rec_merged

In [ ]:
community_summary = (
    terms_with_rec_merged
    .groupby("community_id")["recurrence"]
    .agg(mean_recurrence="mean", term_count="count")
    .reset_index()
)

print(community_summary)

In [ ]:
display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [ ]:
for c in communities:
    print(len(c))

In [ ]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [ ]:
all_comms_ncbi = index_to_ncbi(communities,index_to_gene_distinct)

In [ ]:
print(all_comms_ncbi)

In [ ]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))

In [ ]:
for c in all_comms_ncbi:
    print(len(c),DGIDB_count(c))

In [ ]:
def tbd(id):
    print(len(communities[id]))
    c8_ncbi = index_to_ncbi([communities[id]])[0]
    print(len(c8_ncbi))
    print(DGIDB_count(c8_ncbi))

In [ ]:
def tbd_selected(id):
    print(len(communities_selected[id]))
    c8_ncbi = index_to_ncbi([communities_selected[id]])[0]
    print(len(c8_ncbi))
    print(DGIDB_count(c8_ncbi))

In [ ]:
tbd_selected(1)